In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np
import time

print("Libraries imported successfully!")


Libraries imported successfully!


In [3]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
import time

# Base URL for scraping
base_url = "http://books.toscrape.com/catalogue/page-{}.html"

books_data = []

print("Starting web scraping...")
for page in range(1, 6):  # Scrapes first 5 pages (100 books)
    url = base_url.format(page)
    response = requests.get(url)
    
    if response.status_code != 200:
        print(f"Failed to retrieve page {page}")
        continue
        
    soup = BeautifulSoup(response.text, 'html.parser')
    books = soup.find_all('article', class_='product_pod')
    
    for book in books:
        # 1. Extract Title
        title = book.h3.a['title']
        
        # 2. Extract Price (Regex cleans non-numeric characters like Â or £)
        price_text = book.find('p', class_='price_color').text
        clean_price = re.sub(r'[^0-9.]', '', price_text)
        price = float(clean_price)
        
        # 3. Extract Rating
        rating_class = book.find('p', class_='star-rating')['class']
        rating_map = {'One': 1, 'Two': 2, 'Three': 3, 'Four': 4, 'Five': 5}
        rating = rating_map.get(rating_class[1], 0)
        
        # 4. Extract Availability
        availability = book.find('p', class_='instock availability').text.strip()
        
        # 5. Extract Detail Link
        link = "http://books.toscrape.com/catalogue/" + book.h3.a['href']
        
        books_data.append({
            'Title': title,
            'Price_GBP': price,
            'Rating': rating,
            'Availability': availability,
            'Product_URL': link
        })
        
    print(f"Successfully scraped Page {page}")
    time.sleep(0.5)

# Convert to Pandas DataFrame
df = pd.DataFrame(books_data)
print(f"\nScraping Complete! Total items scraped: {len(df)}")

Starting web scraping...
Successfully scraped Page 1
Successfully scraped Page 2
Successfully scraped Page 3
Successfully scraped Page 4
Successfully scraped Page 5

Scraping Complete! Total items scraped: 100


In [4]:
# Display first few rows
display(df.head())

# Display basic summary statistics
print("\nDataset Summary:")
print(df.describe())

# Save to CSV file
csv_filename = "scraped_books_data.csv"
df.to_csv(csv_filename, index=False)
print(f"\nData saved successfully to {csv_filename}!")


,Title,Price_GBP,Rating,Availability,Product_URL
0,A Light in the Attic,51.77,3,In stock,http://books.toscrape.com/catalogue/a-light-in...
1,Tipping the Velvet,53.74,1,In stock,http://books.toscrape.com/catalogue/tipping-th...
2,Soumission,50.10,1,In stock,http://books.toscrape.com/catalogue/soumission...
3,Sharp Objects,47.82,4,In stock,http://books.toscrape.com/catalogue/sharp-obje...
4,Sapiens: A Brief History of Humankind,54.23,5,In stock,http://books.toscrape.com/catalogue/sapiens-a-...



Dataset Summary:
        Price_GBP      Rating
count  100.000000  100.000000
mean    34.560700    2.930000
std     14.638531    1.423149
min     10.160000    1.000000
25%     19.897500    2.000000
50%     34.775000    3.000000
75%     47.967500    4.000000
max     58.110000    5.000000

Data saved successfully to scraped_books_data.csv!
